In [1]:
import numpy as np
from scipy.stats import norm

## Problem 1

### (a)

Write $X_T = \log S_0 + (r - \tfrac12\sigma^2)T + \Sigma W_T$, where $\Sigma=\mathrm{diag}(\sigma_{[1]},\sigma_{[2]})$ and $W_T=(W^{[1]}_T,W^{[2]}_T)^\top$. Only the last term is random, so
$$
\mathrm{Cov}(X_T)
= \mathbb{E}(\Sigma W_T W_T^\top \Sigma^\top)
= \Sigma\,\mathrm{Cov}(W_T)\,\Sigma^\top
= T\,\Sigma\,\mathrm{Corr}(W_T)\,\Sigma^\top
= \begin{pmatrix}\sigma_{[1]}^2 T & \rho\sigma_{[1]}\sigma_{[2]}T\\ \rho\sigma_{[1]}\sigma_{[2]}T & \sigma_{[2]}^2 T\end{pmatrix}.
$$
With $\sigma_{[1]}=0.3,\sigma_{[2]}=0.2,\rho=0.8,T=1$,
$$
\mathrm{Cov}(X_T)=\begin{pmatrix}0.09 & 0.048\\ 0.048 & 0.04\end{pmatrix}.
$$

In [2]:
class MultiGBM:

    def __init__(self,S0,r,correlations,sigma):
        self.S0 = S0
        self.r = r
        self.correlations = correlations
        self.sigma = sigma

In [3]:
hw6p1dynamics = MultiGBM(S0=np.array([100,110]),r=0.05,
                         correlations = np.array([[1,0.8],[0.8,1]]),
                         sigma = np.diag([0.3, 0.2]))

In [4]:
class CallOnBasket:

    def __init__(self,K,T,weights):
        self.K = K
        self.T = T
        self.weights = weights

In [5]:
hw6p1contract=CallOnBasket(K=110,T=1.0,weights = np.array([1/2, 1/2]))

In [6]:
def CBS(S, t, K, T, q, r, sigma):
    tau = T - t
    d1 = (np.log(S/K) + (r - q + 0.5*sigma**2)*tau) / (sigma*np.sqrt(tau))
    d2 = d1 - sigma*np.sqrt(tau)
    return S*np.exp(-q*tau)*norm.cdf(d1) - K*np.exp(-r*tau)*norm.cdf(d2)


class MCengine:

    def __init__(self, M, antithetic, control, seed):
        self.M = M
        self.antithetic = antithetic
        self.control = control
        self.rng = np.random.default_rng(seed=seed)

    def price_callonbasket_multiGBM(self, contract, dynamics):

        S0, r, sigma, corr = dynamics.S0, dynamics.r, dynamics.sigma, dynamics.correlations
        K, T, w = contract.K, contract.T, contract.weights
        M = self.M

        vols = np.diag(sigma)
        cov = T * (sigma @ corr @ sigma)
        mu  = np.log(S0) + (r - 0.5*vols**2)*T

        Z = self.rng.multivariate_normal(np.zeros(len(S0)), cov, size=M)
        X = mu + Z
        S = np.exp(X)
        H = S @ w
        Y = np.exp(-r*T) * np.maximum(H - K, 0.0)

        if self.antithetic:
            X_a = mu - Z
            S_a = np.exp(X_a)
            H_a = S_a @ w
            Y_a = np.exp(-r*T) * np.maximum(H_a - K, 0.0)
            Y_av = 0.5*(Y + Y_a)
            return (Y_av.mean(), Y_av.std(ddof=1)/np.sqrt(M))

        if self.control:
            G = np.exp(X @ w)
            Ystar = np.exp(-r*T) * np.maximum(G - K, 0.0)

            s1, s2, rho = vols[0], vols[1], corr[0,1]
            sG = 0.5*np.sqrt(s1**2 + 2*rho*s1*s2 + s2**2)
            qG = (s1**2 + s2**2 - 2*rho*s1*s2) / 8.0
            S0G = np.sqrt(S0[0]*S0[1])
            Cstar = CBS(S0G, 0.0, K, T, qG, r, sG)

            beta = np.cov(Y, Ystar, ddof=1)[0,1] / np.var(Ystar, ddof=1)
            Y_cv = Y + beta*(Cstar - Ystar)
            return (Y_cv.mean(), Y_cv.std(ddof=1)/np.sqrt(M))

        return (Y.mean(), Y.std(ddof=1)/np.sqrt(M))

In [7]:
hw6p1bMC=MCengine(M=10000,antithetic=False,control=False,seed=0)
(call_price_ordinary, std_err_ordinary) = hw6p1bMC.price_callonbasket_multiGBM(hw6p1contract,hw6p1dynamics)
print(call_price_ordinary, std_err_ordinary)

9.875007332598607 0.1683896056954233


In [8]:
hw6p1cMC=MCengine(M=10000,antithetic=True,control=False,seed=0)
(call_price_AV, std_err_AV) = hw6p1cMC.price_callonbasket_multiGBM(hw6p1contract,hw6p1dynamics)
print(call_price_AV, std_err_AV)

9.94112718062444 0.0951845020035323


### (d)

By Itô (or directly from $S^{[j]}_T = S^{[j]}_0 \exp\{(r-\tfrac12\sigma_{[j]}^2)T+\sigma_{[j]}W^{[j]}_T\}$),
$$
\log G_T = \tfrac12\big(\log S^{[1]}_T+\log S^{[2]}_T\big)
= \tfrac12\log(S^{[1]}_0 S^{[2]}_0) + \Big(r-\tfrac{\sigma_{[1]}^2+\sigma_{[2]}^2}{4}\Big)T + \tfrac12\big(\sigma_{[1]}W^{[1]}_T+\sigma_{[2]}W^{[2]}_T\big).
$$
Only the last term is random, so
$$
\mathbb{E}\log G_T = \tfrac12\log(S^{[1]}_0 S^{[2]}_0) + \Big(r-\tfrac{\sigma_{[1]}^2+\sigma_{[2]}^2}{4}\Big)T,
$$
and using $\mathrm{Cov}(W^{[1]}_T,W^{[2]}_T)=\rho T$,
$$
\mathrm{Var}\log G_T = \tfrac14\big(\sigma_{[1]}^2+2\rho\sigma_{[1]}\sigma_{[2]}+\sigma_{[2]}^2\big)T.
$$

### (e)

Match the lognormal distribution above to a GBM with continuous yield $q$:
$$
\log S_T - \log S_0 = (r-q-\tfrac12\sigma^2)T + \sigma W_T.
$$
Reading off variance and drift gives
$$
S^{(G)}_0 = \sqrt{S^{[1]}_0 S^{[2]}_0},\qquad
\sigma_G = \tfrac12\sqrt{\sigma_{[1]}^2 + 2\rho\sigma_{[1]}\sigma_{[2]} + \sigma_{[2]}^2},\qquad
q_G = \tfrac{\sigma_{[1]}^2 + \sigma_{[2]}^2 - 2\rho\sigma_{[1]}\sigma_{[2]}}{8}.
$$
Therefore
$$
\boxed{\;C_G \;=\; C_{BS}\big(\sqrt{S^{[1]}_0 S^{[2]}_0},\ 0,\ K,\ T,\ \tfrac{\sigma_{[1]}^2+\sigma_{[2]}^2-2\rho\sigma_{[1]}\sigma_{[2]}}{8},\ r,\ \tfrac12\sqrt{\sigma_{[1]}^2+2\rho\sigma_{[1]}\sigma_{[2]}+\sigma_{[2]}^2}\big).\;}
$$
With $\sigma_{[1]}=0.3,\sigma_{[2]}=0.2,\rho=0.8$: $\sigma_G=\sqrt{0.0565}\approx 0.23770$, $q_G=0.00425$, $S^{(G)}_0=\sqrt{11000}\approx 104.881$.

In [9]:
hw6p1fMC=MCengine(M=10000,antithetic=False,control=True,seed=0)
(call_price_CV, std_err_CV) = hw6p1fMC.price_callonbasket_multiGBM(hw6p1contract,hw6p1dynamics)
print(call_price_CV, std_err_CV)

9.991036492142719 0.004401017044434686
